## 📑 HAPS L3869 Perimeter
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYAUH)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication  

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
# env_prefix = 'p'
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]
 
dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d')   

###### Schema of the table

In [0]:
table_name = 'HAPS_L3869Perimeter'

schema = (
    "BusinessDate date,"
    "ExposureKey string,"
    "AgreementKey string,"
    "CDI string,"
    "PartyTaxRegistrationNumber string,"
    "PartyRelationshipType string,"
    "PortfolioName string,"
    "AssetSegment string,"
    "ExposureStatus string,"
    "ExposureStatusCalculated string,"
    "SourceBucket int,"
    "BucketCalculated int,"
    "OnBalanceAmount decimal(19,2),"
    "OnBalanceAmountEUR decimal(19,2),"
    "GBV decimal(19,2),"
    "GBVEUR decimal(19,2),"
    "LawStatus string,"
    "LawDecision string,"
    "SubmissionDate date,"
    "CourtDate date,"
    "L3869InterbankFlag int,"
    "L3869InterbankDescription string,"
    "InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### haps_portfolios

In [0]:
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_L3869Perimeter'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")

##### dexposure

In [0]:
dexposure = spark.sql(f'''
SELECT de.* 
,cast(null as int) as SourceBucket
FROM `{env_prefix}-cdh`.derivative.dexposure de 
INNER JOIN haps_portfolios AS A
ON de.PortfolioName = A.PortfolioName
WHERE
    sourcesystem = 'profits' 
    AND businessdate = {run_date!r}
    AND IsActiveFlag = TRUE
    -- AND PartyRelationshipType = 'Primary Owner'
    AND (
        (
            de.PortfolioName = 'Cosmos Retail' AND
            (
                (
                    ProductKey NOT LIKE '41000;31%' AND 
                    ProductKey NOT LIKE '41002%' AND
                    ProductKey NOT LIKE '41006%' AND
                    ProductKey NOT LIKE '41005;31%' AND
                    ProductKey NOT LIKE '40584%' AND
                    ProductKey NOT LIKE '40011%' AND
                    ProductKey NOT LIKE '40586;31%' AND
                    ProductKey NOT LIKE '40010;40010'
                ) 
                OR 
                ProductKey IN ('41000;3118','41005;3116','41005;3151')
            )
        )
        OR
        (
            de.PortfolioName IN ('Galaxy II Funding DAC','Orion X Securitisation DAC') AND
            (
                (
                    ProductKey NOT LIKE '40515;31%' AND 
                    ProductKey NOT LIKE '40565;31%' AND
                    ProductKey NOT LIKE '40584%' AND
                    ProductKey NOT LIKE '40011%' AND
                    ProductKey NOT LIKE '40586;31%' AND
                    ProductKey NOT LIKE '40324;4015' AND
                    ProductKey NOT LIKE '40010;40010'
                ) 
                OR 
                ProductKey IN ('40515;3118','40565;3116','40565;3151')
            )
        )
    )
''')

dexposure.createOrReplaceTempView("dexposure")

In [0]:
from pyspark.sql.types import IntegerType
from pyspark.sql.types import StringType
from pyspark.sql.types import BooleanType
from pyspark.sql.functions import udf

def get_bucket_from_dpd( dpd : int ):
        
    if dpd is None:
        return None
    elif dpd <= 0:
        return 0
    elif 1 <= dpd <= 29:
        return 1
    elif 30 <= dpd <= 59:
        return 2
    elif 60 <= dpd <= 89:
        return 3
    elif 90 <= dpd <= 119:
        return 4
    elif 120 <= dpd <= 149:
        return 5
    elif 150 <= dpd <= 179:
        return 6
    elif 180 <= dpd <= 209:
        return 7
    elif 210 <= dpd <= 239:
        return 8
    elif 240 <= dpd <= 269:
        return 9
    elif 270 <= dpd <= 299:
        return 10
    elif 300 <= dpd <= 329:
        return 11
    elif 330 <= dpd <= 359:
        return 12
    elif 360 <= dpd <= 720:
        return 13
    elif dpd > 720:
        return 14


# Convert your Python function into a UDF
get_bucket_from_dpd_udf = udf(get_bucket_from_dpd, IntegerType())



def is_valid_tax_number(Input : str):
    if Input is None:
        return None
    elif not Input.isdigit():
        return None
    elif Input == '999999993':
        return None
    elif any(c in Input for c in ['+', '-', '.', ',', 'E', 'e']):
        return None 
    elif len(Input) == 8:
        Input = '0' + Input
    if len(Input) != 9:
        return None
    factor = 256
    check_digit = int(Input[8])
    value =0
    for i in range(0,8):
        digit = int(Input[i])
        value = value + digit * factor
        factor = factor // 2
    if value == 0:
        return False
    checksum = value % 11
    if checksum == 10:
        checksum=0
    if checksum == check_digit:
        return True
    else:
        return False

# Convert your Python function into a UDF
is_valid_tax_number_udf = udf(is_valid_tax_number, BooleanType())

dexposure = dexposure.withColumn("SourceBucket", get_bucket_from_dpd_udf(dexposure["DaysPastDueWithMateriality"]))

##### exposure

In [0]:
exposure = spark.sql(f'''
SELECT *
FROM `{env_prefix}-cdh`.primary.exposure
WHERE BusinessDate={run_date!r} 
AND IsActiveFlag = true 
''')
exposure.createOrReplaceTempView("exposure")

##### casecode

In [0]:
casecode = spark.sql(f'''
SELECT *
FROM `{env_prefix}-cdh`.primary.casecode
WHERE BusinessDate={run_date!r} 
''')
casecode.createOrReplaceTempView("casecode")

##### Perimeter

In [0]:
Perimeter = spark.sql(f'''
SELECT			
     de.Exposurekey AS Exposurekey
    ,de.ExposureStatus AS ExposureStatus
    ,CASE 
        WHEN de.ExposureStatus = 'Writen Off'                       THEN 'Writen-Off'
        WHEN de.ExposureStatus = 'Define Delay'                     THEN 'Denounced'
        WHEN de.ExposureStatus = 'Overdue' AND de.SourceBucket = 0  THEN 'Current'
        WHEN de.ExposureStatus = 'Overdue'                          THEN 'NonDenounced'
        WHEN de.ExposureStatus = 'Normal'                           THEN 'Current'
        ELSE 'Check'
     END AS ExposureStatusCalculated
    ,de.SourceBucket AS SourceBucket
    ,CASE 
        WHEN de.ExposureStatus = 'Writen Off' THEN 99
        WHEN de.ExposureStatus = 'Define Delay' AND de.AssetSegment IN ('SB', 'ML') THEN 13
        ELSE CASE WHEN de.SourceBucket = 14 THEN 13 ELSE de.SourceBucket END
     END AS BucketInt
    ,de.AssetSegment
    ,de.PortfolioName
    ,de.OnBalance AS OnBalanceAmount
    ,`{env_prefix}-cdh`.derivative.converttoeur(de.Currency, de.OnBalance, de.BusinessDate) AS OnBalanceAmountEUR
    ,de.GBV AS GBV
    ,`{env_prefix}-cdh`.derivative.converttoeur(de.Currency, de.GBV, de.BusinessDate) AS GBVEUR
    ,de.DaysPastDueWithMateriality
FROM dexposure de
-- LEFT JOIN SourceBucket s ON s.BusinessDate = de.BusinessDate AND s.ExposureKey = de.ExposureKey AND s.SourceSystem = 'profits'
LEFT JOIN casecode cc ON cc.businessdate = de.BusinessDate AND cc.sourcesystem = 'qca' AND cc.casecodekey = de.CaseCode
WHERE cc.Queue NOT LIKE '%Υπό%Αίρεση%'
''')
Perimeter.createOrReplaceTempView("Perimeter")

##### UnderLaw

In [0]:
UnderLaw = spark.sql(
f'''
    SELECT * FROM (
        SELECT  *,
                RANK() OVER (PARTITION BY CDI ORDER BY BusinessDate DESC) Rank
        FROM `{env_prefix}-cdh`.primary.underlaw
        WHERE BusinessDate <= {run_date!r}
    ) a WHERE Rank = 1
'''
)
UnderLaw.createOrReplaceTempView("UnderLaw")

##### L3869Perimeter

In [0]:
L3869Perimeter = spark.sql(f"""
SELECT DISTINCT     
        cast({run_date!r}  as  date)                         AS BusinessDate,
        a.Exposurekey                                        AS Exposurekey,
        a.AgreementKey                                       AS AgreementKey,                                
        a.CDI                                                AS CDI,
        a.PartyTaxRegistrationNumber                         AS PartyTaxRegistrationNumber,
        a.PartyRelationshipType                              AS PartyRelationshipType,
        a.PortfolioName                                      AS PortfolioName,
        MT.AssetSegment                                      AS AssetSegment,
        MT.ExposureStatus                                    AS ExposureStatus,
        MT.ExposureStatusCalculated                          AS ExposureStatusCalculated,
        MT.DaysPastDueWithMateriality                        AS DaysPastDueWithMateriality,
        MT.SourceBucket                                      AS SourceBucket,
        MT.BucketInt                                         AS BucketCalculated,
        CAST(MT.OnBalanceAmount      AS DECIMAL(19,2))       AS OnBalanceAmount,
        CAST(MT.OnBalanceAmountEUR   AS DECIMAL(19,2))       AS OnBalanceAmountEUR,
        CAST(MT.GBV                  AS DECIMAL(19,2))       AS GBV,
        CAST(MT.GBVEUR               AS DECIMAL(19,2))       AS GBVEUR,  
        UL.LawStatus                                         AS LawStatus,
        CASE 
            WHEN LawStatus = 'Έφεση (Πρωτόδικη Απόφ. Δεκτή)' THEN 'Positive Court Decision' 
            ELSE CalcLawStatus 
        END                                                  AS LawDecision,
        UL.SubmissionDate                                    AS SubmissionDate,
        UL.CourtDate                                         AS CourtDate,
        CASE 
            WHEN ul.InterbankStatus3869 NOT IN (
                'N/A',
                'Πελάτης Διατραπεζικής Ν3869 Follower',
                'Πελάτης Διατραπεζικής Ν3869 Leader',
                'Τειρεσίας - Παύση Εκκρεμοδικίας',
                'Έξοδος από Διατραπεζική ν.3869'
            ) THEN 1 ELSE 0 
        END                                                  AS L3869InterbankFlag,
        CASE 
            WHEN ul.InterbankStatus3869 NOT IN (
                'N/A',
                'Πελάτης Διατραπεζικής Ν3869 Follower',
                'Πελάτης Διατραπεζικής Ν3869 Leader',
                'Τειρεσίας - Παύση Εκκρεμοδικίας',
                'Έξοδος από Διατραπεζική ν.3869'
            ) THEN InterbankStatus3869 ELSE NULL 
        END                                                  AS L3869InterbankDescription

FROM exposure a
INNER JOIN Perimeter MT 
        ON a.Exposurekey = MT.Exposurekey
LEFT JOIN UnderLaw UL 
        ON a.CDI = UL.CDI
WHERE a.BusinessDate={run_date!r} 
AND a.IsActiveFlag = true 
AND (UL.UnderLawType = 'N3869' OR UL.UnderLawType = 'N3869') 
AND UL.InLawFlag = 1
""")
L3869Perimeter.createOrReplaceTempView("L3869Perimeter")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
  CAST({run_date!r}                       AS date)          AS BusinessDate  
, CAST(ExposureKey                        AS string)        AS ExposureKey  
, CAST(AgreementKey                       AS string)        AS AgreementKey  
, CAST(CDI                                AS string)        AS CDI  
, CAST(PartyTaxRegistrationNumber         AS string)        AS PartyTaxRegistrationNumber  
, CAST(PartyRelationshipType              AS string)        AS PartyRelationshipType  
, CAST(PortfolioName                      AS string)        AS PortfolioName  
, CAST(AssetSegment                       AS string)        AS AssetSegment  
, CAST(ExposureStatus                     AS string)        AS ExposureStatus  
, CAST(ExposureStatusCalculated           AS string)        AS ExposureStatusCalculated  
, CAST(SourceBucket                       AS int)           AS SourceBucket  
, CAST(BucketCalculated                   AS int)           AS BucketCalculated  
, CAST(OnBalanceAmount                    AS decimal(19,2)) AS OnBalanceAmount  
, CAST(OnBalanceAmountEUR                 AS decimal(19,2)) AS OnBalanceAmountEUR  
, CAST(GBV                                AS decimal(19,2)) AS GBV  
, CAST(GBVEUR                             AS decimal(19,2)) AS GBVEUR  
, CAST(LawStatus                          AS string)        AS LawStatus  
, CAST(LawDecision                        AS string)        AS LawDecision  
, CAST(SubmissionDate                     AS date)          AS SubmissionDate  
, CAST(CourtDate                          AS date)          AS CourtDate  
, CAST(L3869InterbankFlag                 AS int)           AS L3869InterbankFlag  
, CAST(L3869InterbankDescription          AS string)        AS L3869InterbankDescription
, CAST(getdate()                          AS timestamp)     AS InsertTimeStamp
FROM L3869Perimeter
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey", "CDI"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
L3869Perimeter = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
L3869Perimeter.createOrReplaceTempView("L3869Perimeter")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/L3869Perimeter/" 
file_name = "L3869Perimeter_" +run_date+ ".csv"
L3869Perimeter.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") 
rename_and_cleanup_files(output_path, file_name)